In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Load Dataset
df = pd.read_csv("Dataset .csv")
df.columns = df.columns.str.strip()

# 2. Data Preprocessing
# Drop rows with missing values in key columns
df = df.dropna(subset=['Cuisines', 'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes'])

# Extract primary cuisine as target variable (since restaurants can have multiple listed)
df['Primary_Cuisine'] = df['Cuisines'].apply(lambda x: str(x).split(',')[0].strip())

# Select top N most common cuisines to keep classification clean and balanced
top_cuisines = df['Primary_Cuisine'].value_counts().nlargest(10).index
df_filtered = df[df['Primary_Cuisine'].isin(top_cuisines)].copy()

# Encode features and target variable
le_city = LabelEncoder()
df_filtered['City_Encoded'] = le_city.fit_transform(df_filtered['City'].astype(str))

le_target = LabelEncoder()
df_filtered['Target'] = le_target.fit_transform(df_filtered['Primary_Cuisine'])

# Define Features (X) and Target (y)
feature_cols = ['Average Cost for two', 'Price range', 'Aggregate rating', 'Votes', 'City_Encoded']
X = df_filtered[feature_cols]
y = df_filtered['Target']

# 3. Split Dataset into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train Random Forest Classifier Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make Predictions
y_pred = model.predict(X_test)

# 5. Evaluate Performance
accuracy = accuracy_score(y_test, y_pred)
print(f"--- Model Overall Accuracy: {accuracy * 100:.2f}% ---\n")

print("--- Classification Metric Report (Precision, Recall, F1-Score) ---")
target_names = le_target.classes_
print(classification_report(y_test, y_pred, target_names=target_names))

# 6. Analyze Feature Importance & Bias
print("--- Feature Importance Breakdown ---")
feature_importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(feature_importances)

--- Model Overall Accuracy: 42.69% ---

--- Classification Metric Report (Precision, Recall, F1-Score) ---
              precision    recall  f1-score   support

    American       0.54      0.61      0.57        56
      Bakery       0.20      0.14      0.16       124
        Cafe       0.31      0.28      0.29       123
     Chinese       0.20      0.13      0.16       171
 Continental       0.23      0.15      0.18        47
   Fast Food       0.28      0.15      0.19       135
      Mithai       0.29      0.29      0.29        49
North Indian       0.54      0.72      0.62       599
South Indian       0.19      0.10      0.13        52
 Street Food       0.25      0.23      0.24        47

    accuracy                           0.43      1403
   macro avg       0.30      0.28      0.28      1403
weighted avg       0.38      0.43      0.39      1403

--- Feature Importance Breakdown ---
Votes                   0.423191
Average Cost for two    0.214681
Aggregate rating        0.20085